In [ ]:
!pip install -q efficientnet
!pip install -q pandas scikit-learn
!pip install tensorflow

In [ ]:
!gdown https://drive.google.com/uc?id=15TIEzQMtwyu6Kzr6HfZO_07N9E4cDTrn

In [ ]:
!unzip -q dataset.zip

In [ ]:
import os
from PIL import Image, UnidentifiedImageError

def clean_image_folder(folder_path):
    """
    Walks through each subfolder in folder_path, attempts to open each file as an image,
    and deletes files that are not valid images.
    """
    for class_name in os.listdir(folder_path):
        class_dir = os.path.join(folder_path, class_name)
        if not os.path.isdir(class_dir):
            continue

        for filename in os.listdir(class_dir):
            file_path = os.path.join(class_dir, filename)
            if os.path.isdir(file_path):
                continue

            try:
                # Try opening and verifying the image
                with Image.open(file_path) as img:
                    img.verify()
            except (UnidentifiedImageError, IOError, SyntaxError):
                print(f"Deleting invalid image: {file_path}")
                os.remove(file_path)

def main():
    base_dir = "dataset"
    for split in ("train", "val"):
        split_dir = os.path.join(base_dir, split)
        if not os.path.isdir(split_dir):
            print(f"Warning: {split_dir} does not exist, skipping.")
            continue
        clean_image_folder(split_dir)

main()

In [ ]:
import json, numpy as np
from pathlib import Path
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (EarlyStopping,
                                        ReduceLROnPlateau,
                                        ModelCheckpoint)

# -------------------------------------------------
# 1.  Paths and basic hyper-parameters
# -------------------------------------------------

In [ ]:
TRAIN_DIR = Path("dataset/train")
VAL_DIR   = Path("dataset/val")

IMG_SIZE   = (260, 260)        # EfficientNet-B2 default
BATCH_SIZE = 16
N_CLASSES  = 6

# ------------------------------------------------------------------
# 2. data generators  (heavy aug on train)
# ------------------------------------------------------------------

In [ ]:
train_gen = ImageDataGenerator(
    rescale            = 1/255.,
    rotation_range     = 20,
    width_shift_range  = 0.2,
    height_shift_range = 0.2,
    shear_range        = 0.2,
    zoom_range         = 0.2,
    horizontal_flip    = True,
    fill_mode          = "nearest")

val_gen   = ImageDataGenerator(rescale = 1/255.)

train_ds = train_gen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical")

val_ds   = val_gen.flow_from_directory(
    VAL_DIR,   target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical")

# save mapping for inference
with open("class_indices.json", "w") as f:
    json.dump(train_ds.class_indices, f, indent=2)

# ------------------------------------------------------------------
# 3. class-weight dictionary  (handles imbalance)
# ------------------------------------------------------------------

In [ ]:
y = train_ds.classes
weights = compute_class_weight(class_weight="balanced",
                               classes=np.unique(y),
                               y=y)
class_weights = dict(enumerate(weights))

NameError: name 'EfficientNetB0' is not defined

# ------------------------------------------------------------------
# 4. model definition
# ------------------------------------------------------------------

In [ ]:
base = EfficientNetB2(weights="imagenet", include_top=False,
                      input_shape=(*IMG_SIZE, 3))
x   = GlobalAveragePooling2D()(base.output)
out = Dense(N_CLASSES, activation="softmax")(x)
model = Model(inputs=base.input, outputs=out)

NameError: name 'base' is not defined

# ------------------------------------------------------------------
# 5. training schedule
# ------------------------------------------------------------------

In [ ]:
cbs = [EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
       ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3),
       ModelCheckpoint("best_efficientnet_cattle_b2.h5",
                       monitor="val_loss", save_best_only=True)]

# Phase 1 – train classifier head
for l in base.layers: l.trainable = False
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=20,
          class_weight=class_weights, callbacks=cbs)

# Phase 2 – fine-tune entire network with low LR
for l in base.layers: l.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="categorical_crossentropy",
              metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=30,
          class_weight=class_weights, callbacks=cbs)

# ------------------------------------------------------------------
# 6. save final weights
# ------------------------------------------------------------------


In [ ]:
model.save("final_efficientnet_cattle_b2.h5")
print("✅ training complete – models saved")